## 01 Data Overview


In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import load_raw_order_book, standardize_lob_columns
from src.data_quality import run_basic_quality_report
from src.features import add_basic_lob_features, add_depth_features, add_return_features
from src.labels import add_future_return_label

### Load Raw Data

In [2]:
raw_path = PROJECT_ROOT / "data" / "raw" / "BTC_1sec.csv"
raw_df = load_raw_order_book(raw_path)
raw_df.head()

,Unnamed: 0,system_time,midpoint,spread,buys,sells,bids_distance_0,bids_distance_1,bids_distance_2,bids_distance_3,...,asks_market_notional_9,asks_market_notional_10,asks_market_notional_11,asks_market_notional_12,asks_market_notional_13,asks_market_notional_14,timestamp,symbol,frequency,source_file
0,0,2021-04-07 11:32:42.122161+00:00,56035.995,0.01,0.0,0.0,-8.922836e-08,-2.676851e-07,-0.00005,-0.000245,...,0.0,0.0,0.0,0.0,0.0,0.0,2021-04-07 11:32:42.122161+00:00,BTC,1sec,BTC_1sec.csv
1,1,2021-04-07 11:32:43.122161+00:00,56035.995,0.01,0.0,0.0,-8.922836e-08,-2.676851e-07,-0.00005,-0.000245,...,0.0,0.0,0.0,0.0,0.0,0.0,2021-04-07 11:32:43.122161+00:00,BTC,1sec,BTC_1sec.csv
2,2,2021-04-07 11:32:44.122161+00:00,56035.995,0.01,0.0,0.0,-8.922836e-08,-2.676851e-07,-0.00005,-0.000245,...,0.0,0.0,0.0,0.0,0.0,0.0,2021-04-07 11:32:44.122161+00:00,BTC,1sec,BTC_1sec.csv
3,3,2021-04-07 11:32:45.122161+00:00,56035.995,0.01,0.0,0.0,-8.922836e-08,-2.676851e-07,-0.00005,-0.000245,...,0.0,0.0,0.0,0.0,0.0,0.0,2021-04-07 11:32:45.122161+00:00,BTC,1sec,BTC_1sec.csv
4,4,2021-04-07 11:32:46.122161+00:00,56035.995,0.01,0.0,0.0,-8.922836e-08,-2.676851e-07,-0.00005,-0.000245,...,0.0,0.0,0.0,0.0,0.0,0.0,2021-04-07 11:32:46.122161+00:00,BTC,1sec,BTC_1sec.csv


### Standardize and Validate

price: bid_p1 = midpoint * (1 + bids_distance_0)  
quantity: bid_q1 = bids_notional_0 

In [3]:
lob = standardize_lob_columns(raw_df, source_file=raw_path.name)
quality_report = run_basic_quality_report(lob, levels=5)
quality_report

Basic quality report
  row count: 1030728
  column count: 64
  timestamp min: 2021-04-07 11:32:42.122161+00:00
  timestamp max: 2021-04-19 09:54:22.386544+00:00
  duplicate timestamp count: 0
  bid_p1 < ask_p1 ratio: 1.0
  positive price ratio top levels: 1.0
  non-negative size ratio top levels: 1.0
  spread summary: {'min': 0.009999999529100023, 'max': 1245.1000185738521, 'mean': 1.3140331202547664, 'median': 0.010000000096624717}
  percentage of 1-second gaps: 0.9999301464595898
  maximum timestamp gap seconds: 30.000617
  number of gaps larger than 1 second: 71
  missing value ratio core columns: {'bid_p1': 0.0, 'ask_p1': 0.0, 'bid_q1': 0.0, 'ask_q1': 0.0, 'bid_p2': 0.0, 'ask_p2': 0.0, 'bid_q2': 0.0, 'ask_q2': 0.0, 'bid_p3': 0.0, 'ask_p3': 0.0, 'bid_q3': 0.0, 'ask_q3': 0.0, 'bid_p4': 0.0, 'ask_p4': 0.0, 'bid_q4': 0.0, 'ask_q4': 0.0, 'bid_p5': 0.0, 'ask_p5': 0.0, 'bid_q5': 0.0, 'ask_q5': 0.0}


{'row_count': 1030728,
 'column_count': 64,
 'missing_value_ratio_core_columns': {'bid_p1': 0.0,
  'ask_p1': 0.0,
  'bid_q1': 0.0,
  'ask_q1': 0.0,
  'bid_p2': 0.0,
  'ask_p2': 0.0,
  'bid_q2': 0.0,
  'ask_q2': 0.0,
  'bid_p3': 0.0,
  'ask_p3': 0.0,
  'bid_q3': 0.0,
  'ask_q3': 0.0,
  'bid_p4': 0.0,
  'ask_p4': 0.0,
  'bid_q4': 0.0,
  'ask_q4': 0.0,
  'bid_p5': 0.0,
  'ask_p5': 0.0,
  'bid_q5': 0.0,
  'ask_q5': 0.0},
 'timestamp_sorted': True,
 'timestamp_min': Timestamp('2021-04-07 11:32:42.122161+0000', tz='UTC'),
 'timestamp_max': Timestamp('2021-04-19 09:54:22.386544+0000', tz='UTC'),
 'duplicate_timestamp_count': 0,
 'duplicate_timestamp_ratio': 0.0,
 'bid_p1_lt_ask_p1_ratio': 1.0,
 'violations': 0,
 'positive_price_ratio_top_levels': 1.0,
 'non_negative_size_ratio_top_levels': 1.0,
 'spread_min': 0.009999999529100023,
 'spread_max': 1245.1000185738521,
 'spread_mean': 1.3140331202547664,
 'spread_median': 0.010000000096624717,
 'non_positive_spread_ratio': 0.0,
 'timestamp_diff_s

### Build Processed Dataset


In [4]:
dataset = (
    lob.pipe(add_basic_lob_features)
       .pipe(add_depth_features, levels=5)
       .pipe(add_return_features)
       .pipe(add_future_return_label, horizon_seconds=30, threshold=0.0001)
)

selected_columns = [
    "timestamp", "symbol", "frequency", "source_file",
    "bid_p1", "bid_q1", "ask_p1", "ask_q1",
    "mid_price", "spread", "relative_spread",
    "bid_depth_5", "ask_depth_5", "total_depth_5", "obi_1", "obi_5",
    "mid_return_1s", "mid_return_5s", "mid_return_10s",
    "future_mid_price_30s", "future_return_30s", "label",
]
existing_columns = [column for column in selected_columns if column in dataset.columns]
dataset = dataset[existing_columns].dropna(subset=["label"]).reset_index(drop=True)
dataset.head()

,timestamp,symbol,frequency,source_file,bid_p1,bid_q1,ask_p1,ask_q1,mid_price,spread,...,ask_depth_5,total_depth_5,obi_1,obi_5,mid_return_1s,mid_return_5s,mid_return_10s,future_mid_price_30s,future_return_30s,label
0,2021-04-07 11:32:42.122161+00:00,BTC,1sec,BTC_1sec.csv,56035.99,3061.860107,56036.0,1902.290039,56035.995,0.01,...,113298.046913,274830.845879,0.233589,0.175507,NaN,NaN,NaN,55867.345,-0.00301,Down
1,2021-04-07 11:32:43.122161+00:00,BTC,1sec,BTC_1sec.csv,56035.99,3061.860107,56036.0,1902.290039,56035.995,0.01,...,113298.046913,274830.845879,0.233589,0.175507,0.0,NaN,NaN,55867.345,-0.00301,Down
2,2021-04-07 11:32:44.122161+00:00,BTC,1sec,BTC_1sec.csv,56035.99,3061.860107,56036.0,1902.290039,56035.995,0.01,...,113298.046913,274830.845879,0.233589,0.175507,0.0,NaN,NaN,55867.345,-0.00301,Down
3,2021-04-07 11:32:45.122161+00:00,BTC,1sec,BTC_1sec.csv,56035.99,3061.860107,56036.0,1902.290039,56035.995,0.01,...,113298.046913,274830.845879,0.233589,0.175507,0.0,NaN,NaN,55867.345,-0.00301,Down
4,2021-04-07 11:32:46.122161+00:00,BTC,1sec,BTC_1sec.csv,56035.99,3061.860107,56036.0,1902.290039,56035.995,0.01,...,113298.046913,274830.845879,0.233589,0.175507,0.0,NaN,NaN,55867.345,-0.00301,Down


In [ ]:
processed_path = PROJECT_ROOT / "data" / "processed" / "obi_dataset.parquet"
processed_path.parent.mkdir(parents=True, exist_ok=True)
dataset.to_parquet(processed_path, index=False)
print(f"Saved {len(dataset):,} rows to {processed_path}")